・データソース
kaggle
https://www.kaggle.com/datasets/jp797498e/twitter-entity-sentiment-analysis

yahooニュース
記事A（だいぶ炎上）
https://news.yahoo.co.jp/articles/9d90914fe398dd5105fc05040ddae5a2dba6fb33

記事B（ちょっと炎上）
https://news.yahoo.co.jp/articles/3148ee46c0f9b4cd27248950bf2322940d8d7e5d

記事C（中立）
https://news.yahoo.co.jp/articles/482e38e1cdc0c2daec9dfefad2f78a968eb1b700/comments



In [ ]:
import os

from dotenv import load_dotenv
from google import genai

1.gemini apiの設定

In [ ]:
load_dotenv()
gemini_api_key = os.getenv("API_KEY")

In [ ]:
# 創造性不要の分類タスクのため、temparatureを低めに
generation_config = {
    "temperature": 0.2,
    "max_output_tokens": 500, 
}

model = "gemini-3-flash-preview" 

In [ ]:
client = genai.Client(api_key = gemini_api_key)

2.プロンプトを試す

In [ ]:
def categorize(prompt):
    contents = prompt

    response = client.models.generate_content(
        model = model, 
        config = generation_config,
        contents = contents
    )
    
    print(response.text)

In [ ]:
# comment_text = "Small businesses are drowning in taxes while large corporations thrive, something is seriously unbalanced."
comment_text ="動作確認！"

In [ ]:
prompt2 = f'''
あなたはSNSコメント分析の専門家です。
以下の日本語または英語のコメントを読み、行政・税金に関する不満の種類を1つだけ分類してください。

【分類カテゴリ】
1. 無駄遣い：税金の使い道・支出内容が不適切／無駄だと批判している
2. 不透明性：意思決定プロセス・説明不足・情報開示に不満がある
3. 不公平：負担や恩恵の偏り、不平等さに不満がある
4. 強い批判：怒り・攻撃的な表現が主軸の不満（※内容より感情が中心）
5. その他：上記に当てはまらない／中立・事実・軽い意見

【重要ルール（優先順位）】
以下の順で判断してください：

Step1：まず「何に対する不満か（内容）」を判断
  - 無駄遣い / 不透明性 / 不公平 のいずれかに該当するか確認

Step2：その上で「感情の強さ」を確認
  - 強い怒り・攻撃表現があり、かつ内容より感情が主軸の場合のみ「強い批判」を選択

※単に怒り表現が含まれるだけでは「強い批判」にしない  
※内容（無駄遣い・不透明性・不公平）が明確な場合はそちらを優先

【判断の補助ルール】
- 「説明不足・不明確・よく分からない」→ 不透明性
- 「格差・偏り・誰かが得している」→ 不公平
- 「金の使い方がおかしい・無駄」→ 無駄遣い
- 「とにかく怒り・罵倒・攻撃が中心」→ 強い批判

【迷った場合の優先順位】
不透明性 ＞ 不公平 ＞ 無駄遣い ＞ 強い批判

【出力ルール】
- 必ず1つだけ選択
- 最も主な不満を選ぶ
- JSON形式のみ（説明文禁止）

【出力形式】
{{"label": "<カテゴリ名>","confidence": <0.0〜1.0の小数>}}

【コメント】
{comment_text}
'''

In [ ]:
categorize(prompt2)

In [ ]:
prompt = f'''
あなたはSNSコメント分析の専門家です。
以下の日本語コメントを読み、行政・税金に関する不満の種類を1つだけ分類してください。

【分類カテゴリ】
1. 無駄遣い：税金の使い道が無駄だと批判している
2. 不透明性：意思決定の遅さ・説明不足・プロセスへの不満
3. 不公平：特定の層や国に対する優遇・不利益など「扱いの差」への不満
4. 強い批判：怒り・攻撃的な表現を含む強い不満
5. その他：上記に当てはまらない

【分類ルール】
- 必ず1つだけ選択すること
- 最も主な不満を選ぶこと（複数ある場合は主軸を判断）
- ラベルは上記5つから完全一致で選択すること
- confidenceは0.0〜1.0の小数で出力すること（例：0.82）
- 出力は必ず1行のJSONで返すこと
- JSONは必ず「}}」で閉じること
- JSON以外の出力は禁止
- 意思決定の遅さや対応の遅延に対する不満は「不透明性」として扱うこと

【出力形式】
{{"label": "不公平", "confidence": 0.82}}

【コメント】
{comment_text}
'''

In [ ]:
# →json指定すると崩れる時がある。LLMの出力では構造の完全性？が担保されてない？
# 複雑になりやすいjsonよりも、構造がわかりやすいcsvにする？